# Week 6 Presentation Brief — Variation B
## 💧 Groundwater Recharge: Cumulative Infiltration
**SCIE1500**

> Work through all parts during the Week 6 lab. Your **10-minute Week 7 presentation** should cover: the infiltration model, antiderivative, total volume, and whether the seasonal target is met
> **What to submit:** every group member must individually upload their own copy of the same completed presentation slides (PDF or PowerPoint) to the LMS after your presentation — this lets your instructor verify who participated.


---
## 📋 Scenario

![Groundwater infiltration rate and cumulative volume vs target](../images/W6B_groundwater.svg)

A managed aquifer recharge (MAR) scheme in the Murray-Darling Basin diverts floodwater. The infiltration rate declines as soil pores fill:

$$I(t) = 20e^{-0.12t} \quad \text{(ML/day)}$$

where $t$ is days into the recharge season. The scheme runs for a **120-day season**. The seasonal target is **1,500 ML**. Water entitlement costs **$1,500/ML**.

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Derive the antiderivative and calculate cumulative recharge volume | ~25 min |
| B | Determine whether the scheme meets its 1,500 ML target | ~20 min |
| C | Find the optimal season length based on marginal infiltration | ~15 min |
| D | Verify the cumulative volume symbolically using SymPy | ~10 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Groundwater infiltration model
def I(t):
    'Infiltration rate (ML/day) at day t.'
    return 20 * np.exp(-0.12 * t)

target_ML   = 1500
season_days = 120
cost_per_ML = 1500

print("Day | Rate (ML/day)")
print("-" * 22)
for t in [0, 10, 30, 60, 120]:
    print(f"  {t:3d} |  {I(t):6.3f}")

---
## Part A: Antiderivative (~25 min)

$$V(t) = \int_0^t 20e^{-0.12\tau}\,d\tau = \frac{20}{-0.12}\left[e^{-0.12\tau}\right]_0^t = \frac{20}{0.12}(1 - e^{-0.12t})$$

$$V(t) = \frac{500}{3}(1 - e^{-0.12t}) \approx 166.7(1 - e^{-0.12t})$$

In [ ]:
# A.1 — Cumulative volume V(t)
def V(t):
    'Cumulative volume (ML) recharged from day 0 to day t.'
    return (20 / 0.12) * (1 - np.exp(-0.12 * t))

print(f"V(10)  = {V(10):.1f} ML")
print(f"V(30)  = {V(30):.1f} ML")
print(f"V(120) = {V(120):.1f} ML")
print(f"Long-run limit = {20/0.12:.1f} ML")

Plot both.

In [ ]:
# A.2 — Plot both
t_vals = np.linspace(0, 120, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t_vals, I(t_vals), "steelblue", lw=2.5)
ax1.fill_between(t_vals, I(t_vals), alpha=0.2, color="steelblue")
ax1.set_xlabel("Day"); ax1.set_ylabel("Infiltration rate (ML/day)")
ax1.set_title("Infiltration Rate I(t)"); ax1.grid(alpha=0.3)

ax2.plot(t_vals, V(t_vals), "darkblue", lw=2.5)
ax2.axhline(target_ML, color="red", ls="--", label=f"Target: {target_ML} ML")
ax2.axhline(20/0.12, color="gray", ls=":", alpha=0.5, label="Limit: 166.7 ML")
ax2.set_xlabel("Day"); ax2.set_ylabel("Cumulative Volume (ML)")
ax2.set_title("Cumulative Recharge V(t)"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Feasibility-first checkpoint

Before deciding how long to run the scheme, test whether the target is achievable under this model.

- Compare the 1,500 ML target with the horizontal long-run limit of about 166.7 ML. What does this comparison establish before any season-length calculation?
- Explain why extending the season alone cannot overcome that limit.
- Name one model component that would need to change to make a much larger target plausible: the initial rate, the decay rate, or the number/area of recharge basins.

---
## Part B: Does the Scheme Meet Its Target? (~20 min)

In [ ]:
# B.1 — Volume at end of season
V_season = V(season_days)
print(f"Volume recharged over {season_days} days: {V_season:.1f} ML")
print(f"Target: {target_ML} ML")
print(f"Target met? {V_season >= target_ML}")
print(f"Shortfall/surplus: {V_season - target_ML:+.1f} ML")
print(f"\nNote: long-run limit is {20/0.12:.1f} ML — target {target_ML} ML {'can' if 20/0.12 >= target_ML else 'CANNOT'} be met.")

Cost of water used.

In [ ]:
# B.2 — Cost of water used
cost_total = V_season * cost_per_ML
print(f"Cost of {V_season:.0f} ML @ ${cost_per_ML}/ML: ${cost_total:,.0f}")

---
## Part C: Optimal Season Length (~15 min)

At what day does the marginal rate of recharge drop to 1 ML/day (not worth running pumps)?

In [ ]:
# C.1 — Day when rate drops to 1 ML/day
# I(t) = 1 → 20 e^{-0.12t} = 1 → t = -ln(1/20)/0.12
t_cutoff = -np.log(1 / 20) / 0.12
V_cutoff = V(t_cutoff)
print(f"Rate drops to 1 ML/day at t = {t_cutoff:.1f} days")
print(f"Volume by then: {V_cutoff:.1f} ML")

---
## Part D: Symbolic Integration with SymPy (~10 min)

In Part A you found $V(t)$ **by hand** as a definite integral with a variable upper limit. Python's **SymPy** library can evaluate exactly this kind of integral automatically — the same **symbolic computation** idea used in Weeks 4–5 for differentiation, just running in reverse — and it's the standard tool for integration in Python throughout this unit.

Run the cell below to have SymPy integrate $I(t)$ symbolically with a variable upper limit $t$, producing $V(t)$ directly, and confirm it matches your by-hand result from Part A.

In [ ]:
# D.1 — Symbolic integration with SymPy (variable upper limit)
import sympy as sp

t_sym, tau_sym = sp.symbols('t tau', positive=True)
I_expr = 20 * sp.exp(-0.12 * tau_sym)

# Definite integral with a symbolic (variable) upper limit t — same as Part A's V(t)
V_expr = sp.integrate(I_expr, (tau_sym, 0, t_sym))
print(f"I(τ) = {I_expr}")
print(f"V(t) = ∫₀ᵗ I(τ) dτ = {V_expr}   ← matches the by-hand result from Part A")

# Evaluate at t = 120 to compare with Part B
V_120 = float(V_expr.subs(t_sym, 120))
print(f"\nV(120) = {V_120:.1f} ML   ← compare to Part B's V(120)")

**Why this matters:** giving `sp.integrate()` a **symbolic upper limit** (`t_sym` instead of a number) produces the accumulation function $V(t)$ directly, with no separate "solve for the constant" step needed — the constant of integration cancels automatically for any definite integral, symbolic or numeric.

---
## ✅ Presentation Checklist (Week 7, 10 minutes)

1. **Problem** (~2 min): Explain why infiltration declines over the season.
2. **Model** (~3 min): Derive $V(t)$ and explain why the long-run limit is 166.7 ML.
3. **Results** (~3 min): Show whether the 1,500 ML target can be met and present the cost analysis.
4. **Recommendation** (~2 min): Recommend a practical season length with supporting evidence.

---
## 📊 Presentation Marking Rubric (20 marks → scaled to 4% of your unit grade)

Your group presentation is graded out of 20 marks (scaled to 4% of your unit grade — each group presents twice, for 8% total). This rubric determines your group mark, which will be the mark for each contributing member unless we are advised otherwise.

| Criterion | Excellent | Good | Developing | Poor |
|---|---|---|---|---|
| **Problem Formulation** (5 marks) | Clear explanation of the real-world problem; audience understands what question is being answered and why it matters (5) | Problem explained but lacks full context or motivation (3–4) | Problem stated but unclear why it's important (1–2) | No clear problem statement (0) |
| **Mathematical Approach** (5 marks) | Correct model/method selected; clear justification for the choice; key equations presented clearly (5) | Correct approach with minor errors; justification present but weak (3–4) | Approach has errors or is poorly justified (1–2) | Wrong method or no mathematical content shown (0) |
| **Results & Interpretation** (6 marks) | Results are correct and clearly presented; findings are connected to a real-world decision, including limitations/trade-offs (6) | Results mostly correct; some interpretation but lacks depth (4–5) | Results unclear, minor errors, or interpretation is minimal — just states numbers (2–3) | Major errors, no results shown, or no interpretation given (0–1) |
| **Communication Quality** (4 marks) | Effective graphs/visuals support the story; all members participate and speak without reading from notes; well-rehearsed and within the 10-minute limit (4) | Visuals adequate; most members participate; slightly over/under time (3) | Visualization ineffective or missing; uneven participation; timing issues (1–2) | No visuals; one person dominates; major timing problems (0) |

**Total: ____ / 20 marks**